# In silico perturbation experiment

In [2]:
import sys

from geneformer import InSilicoPerturber, InSilicoPerturberStats
from geneformer.in_silico_perturber import ISP_device
from geneformer.in_silico_perturber_stats import GENE_NAME_ID_DICTIONARY_FILE
from geneformer.tokenizer import TOKEN_DICTIONARY_FILE

print(f"usse gpu num: {ISP_device}")
print(f"gene to ens dict: {GENE_NAME_ID_DICTIONARY_FILE}")
print(f"token directory : {TOKEN_DICTIONARY_FILE}")

import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

usse gpu num: cuda:0
gene to ens dict: /app/geneformer/dicts/MLM-re_token_dictionary_v1_GeneSymbol_to_EnsemblID.pkl
token directory : /app/geneformer/dicts/MLM-re_token_dictionary_v1.pkl
True
NVIDIA GB10


In [4]:
import numpy as np
from datasets import load_from_disk

# load disease dataset (xxx.dataset)
dataset_name = "/app/data/Mouse-Genecorpus-20M/eval_dataset/in_silico_perturbation/Cop1KO_isp_mouse_tokenize_dataset_v-n1.dataset"


data = load_from_disk(dataset_name)
print(data)
print("total cells: {}".format(len(data["length"])))

disease_types = np.unique(data["disease"])
print(disease_types)
print(disease_types.shape[0])

Dataset({
    features: ['input_ids', 'cell_types', 'organ_major', 'disease', 'length'],
    num_rows: 12441
})
total cells: 12441
['Cop1_KO' 'Cop1_WT']
2


In [10]:
# in silico perturbation in deletion mode to determine genes whose
# deletion in the dilated cardiomyopathy (dcm) state significantly shifts
# the embedding towards non-failing (nf) state


# select_perturb_type: "delete","overexpress","inhibit","activate"

# "delete": delete gene from rank value encoding
# "overexpress": move gene to front of rank value encoding
# "inhibit": move gene to lower quartile of rank value encoding
# "activate": move gene to higher quartile of rank value encoding


select_perturb_type = "delete"

start_state = "Cop1_WT"  # <-- change this
end_state = "Cop1_KO"  # <-- change this
alt_state = []

use_model_type = "Pretrained"  # <-- change this (you have the pretrained model, not a fine-tuned CellClassifier)

genes_to_perturb_list = []

isp = InSilicoPerturber(
    perturb_type=select_perturb_type,
    perturb_rank_shift=None,
    genes_to_perturb=(
        "all" if len(genes_to_perturb_list) == 0 else genes_to_perturb_list
    ),
    combos=0,
    anchor_gene=None,
    model_type=use_model_type,
    num_classes=2,  # <-- 2 classes: Cop1_WT and Cop1_KO
    emb_mode="cell",
    cell_emb_style="mean_pool",
    filter_data=None,
    cell_states_to_model={
        "state_key": "disease",
        "start_state": start_state,
        "goal_state": end_state,
        "alt_states": alt_state,
    },
    max_ncells=200, # original was 2000, reduce to 200 for test
    emb_layer=0,
    forward_batch_size=200,# original was 50, increase 200 for test
    nproc=8, # adapted to DGX spark
)

# ADD THIS LINE:
organ_data = "Cop1KO"

In [ ]:
# outputs intermediate files from in silico perturbation
start_state = start_state.replace(" ", "-")
end_state = end_state.replace(" ", "-")

import os

DIR_NAME = "/app/output/isp_results"
os.makedirs(DIR_NAME, exist_ok=True)


isp.perturb_data(
    "/app/models/mouse-Geneformer/",
    dataset_name,
    DIR_NAME + "/",
    "output_in-silico_SE{}_OR{}_ST{}_EN{}".format(
        select_perturb_type, organ_data, start_state, end_state
    ),
)

Filter (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

In [6]:
ispstats = InSilicoPerturberStats(
    mode="goal_state_shift",
    genes_perturbed="all" if len(genes_to_perturb_list) == 0 else genes_to_perturb_list,
    combos=0,
    anchor_gene=None,
    cell_states_to_model={
        "state_key": "disease",
        "start_state": start_state,
        "goal_state": end_state,
        "alt_states": alt_state,
    },
)

In [11]:
# extracts data from intermediate files and processes stats to output in final .csv

start_state = start_state.replace(" ", "-")
end_state = end_state.replace(" ", "-")

import os

DIR_NAME = "/app/output/ispstats_results"
os.makedirs(DIR_NAME, exist_ok=True)


ispstats.get_stats(
    "/app/output/isp_results/",  # path to input data (matches perturb_data output)
    None,
    DIR_NAME + "/",
    "output_in-silico_SE{}_OR{}_ST{}_EN{}".format(
        select_perturb_type, organ_data, start_state, end_state
    ),
)  # output prefix


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/2048 [00:00<?, ?it/s]

  0%|          | 0/2048 [00:00<?, ?it/s]

/app/geneformer/in_silico_perturber_stats.py:236: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cos_sims_full_df = pd.concat([cos_sims_full_df,cos_sims_df_i])
